In [26]:
import pandas as pd 
import re 
import numpy as np
### notes
# Ad_Hoc_Export_2025_08_28_08_44_37_AM includes training data 
#

In [27]:
def read_tbl():
    df = pd.read_csv("../data/input/Ad_Hoc_Export_2025_08_28_08_44_37_AM.csv")
    df = df[~(df["Person Student ID"].fillna("") == "")]
    df = df[~(df["Employment Start Date"].fillna("") == "")]
    return df 


def normalize_employment_data(df):
    """
    Normalizes employment data using actual column patterns from the dataset.
    Creates one row per person-employment combination.
    
    Valid employment records must have both 'Employing Organization Name' and 'Employment Start Date'.
    """
    
    # Person identifying columns (appear once per person)
    person_cols = [
        'Person Student ID',
        'Person First Name', 
        'Person Middle Name',
        'Person Last Name',
        'Person Suffix',
        'Person Gender',
        'Person Referencing Gender',
        'Person Status'
    ]
    
    # Define employment record groups based on the actual column patterns
    employment_groups = [
        # First employment (no suffix)
        {
            'Employment Start Date': 'Employment Start Date',
            'Employment End Date': 'Employment End Date',
            'Employment Appointment Type': 'Employment Appointment Type',
            'Employment Employment Assignment': 'Employment Employment Assignment',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)',
            'Employment Status': 'Employment Status',
            'Employment Change Reason': 'Employment Change Reason',
            'Employment Change Comment': 'Employment Change Comment',
            'Is Primary Employment': 'Is Primary Employment',
            'Employing Organization Name': 'Employing Organization Name',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI'
        },
        # Second employment (17-27 suffix range)
        {
            'Employment Start Date': 'Employment Start Date17',
            'Employment End Date': 'Employment End Date18',
            'Employment Appointment Type': 'Employment Appointment Type19',
            'Employment Employment Assignment': 'Employment Employment Assignment20',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)21',
            'Employment Status': 'Employment Status22',
            'Employment Change Reason': 'Employment Change Reason23',
            'Employment Change Comment': 'Employment Change Comment24',
            'Is Primary Employment': 'Is Primary Employment25',
            'Employing Organization Name': 'Employing Organization Name26',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI27'
        },
        # Third employment (48-58 suffix range)
        {
            'Employment Start Date': 'Employment Start Date48',
            'Employment End Date': 'Employment End Date49',
            'Employment Appointment Type': 'Employment Appointment Type50',
            'Employment Employment Assignment': 'Employment Employment Assignment51',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)52',
            'Employment Status': 'Employment Status53',
            'Employment Change Reason': 'Employment Change Reason54',
            'Employment Change Comment': 'Employment Change Comment55',
            'Is Primary Employment': 'Is Primary Employment56',
            'Employing Organization Name': 'Employing Organization Name57',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI58'
        },
        # Fourth employment (79-89 suffix range)
        {
            'Employment Start Date': 'Employment Start Date79',
            'Employment End Date': 'Employment End Date80',
            'Employment Appointment Type': 'Employment Appointment Type81',
            'Employment Employment Assignment': 'Employment Employment Assignment82',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)83',
            'Employment Status': 'Employment Status84',
            'Employment Change Reason': 'Employment Change Reason85',
            'Employment Change Comment': 'Employment Change Comment86',
            'Is Primary Employment': 'Is Primary Employment87',
            'Employing Organization Name': 'Employing Organization Name88',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI89'
        },
        # Fifth employment (110-120 suffix range)
        {
            'Employment Start Date': 'Employment Start Date110',
            'Employment End Date': 'Employment End Date111',
            'Employment Appointment Type': 'Employment Appointment Type112',
            'Employment Employment Assignment': 'Employment Employment Assignment113',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)114',
            'Employment Status': 'Employment Status115',
            'Employment Change Reason': 'Employment Change Reason116',
            'Employment Change Comment': 'Employment Change Comment117',
            'Is Primary Employment': 'Is Primary Employment118',
            'Employing Organization Name': 'Employing Organization Name119',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI120'
        },
        # Sixth employment (141-151 suffix range)
        {
            'Employment Start Date': 'Employment Start Date141',
            'Employment End Date': 'Employment End Date142',
            'Employment Appointment Type': 'Employment Appointment Type143',
            'Employment Employment Assignment': 'Employment Employment Assignment144',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)145',
            'Employment Status': 'Employment Status146',
            'Employment Change Reason': 'Employment Change Reason147',
            'Employment Change Comment': 'Employment Change Comment148',
            'Is Primary Employment': 'Is Primary Employment149',
            'Employing Organization Name': 'Employing Organization Name150',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI151'
        },
        # Seventh employment (172-182 suffix range)
        {
            'Employment Start Date': 'Employment Start Date172',
            'Employment End Date': 'Employment End Date173',
            'Employment Appointment Type': 'Employment Appointment Type174',
            'Employment Employment Assignment': 'Employment Employment Assignment175',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)176',
            'Employment Status': 'Employment Status177',
            'Employment Change Reason': 'Employment Change Reason178',
            'Employment Change Comment': 'Employment Change Comment179',
            'Is Primary Employment': 'Is Primary Employment180',
            'Employing Organization Name': 'Employing Organization Name181',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI182'
        },
        # Eighth employment (198-208 suffix range)
        {
            'Employment Start Date': 'Employment Start Date198',
            'Employment End Date': 'Employment End Date199',
            'Employment Appointment Type': 'Employment Appointment Type200',
            'Employment Employment Assignment': 'Employment Employment Assignment201',
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)202',
            'Employment Status': 'Employment Status203',
            'Employment Change Reason': 'Employment Change Reason204',
            'Employment Change Comment': 'Employment Change Comment205',
            'Is Primary Employment': 'Is Primary Employment206',
            'Employing Organization Name': 'Employing Organization Name207',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI208'
        },
        # Ninth employment (218-227 suffix range)
        {
            'Employment Start Date': 'Employment Start Date218',
            'Employment End Date': 'Employment End Date219',
            'Employment Appointment Type': 'Employment Appointment Type220',
            'Employment Employment Assignment': None,  # Not in column list
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)221',
            'Employment Status': 'Employment Status222',
            'Employment Change Reason': 'Employment Change Reason223',
            'Employment Change Comment': 'Employment Change Comment224',
            'Is Primary Employment': 'Is Primary Employment225',
            'Employing Organization Name': 'Employing Organization Name226',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI227'
        },
        # Tenth employment (237-245 suffix range)
        {
            'Employment Start Date': 'Employment Start Date237',
            'Employment End Date': 'Employment End Date238',
            'Employment Appointment Type': 'Employment Appointment Type239',
            'Employment Employment Assignment': None,  # Not in column list
            'Employment Title/Rank (Current)': 'Employment Title/Rank (Current)240',
            'Employment Status': 'Employment Status241',
            'Employment Change Reason': 'Employment Change Reason242',
            'Employment Change Comment': None,  # Not in column list
            'Is Primary Employment': 'Is Primary Employment243',
            'Employing Organization Name': 'Employing Organization Name244',
            'Employing Organization ORI or WA UBI': 'Employing Organization ORI or WA UBI245'
        }
    ]
    
    results = []
    
    # Process each employment group
    for group in employment_groups:
        employment_data = {}
        
        # Map each employment field to its actual column
        for field, actual_col in group.items():
            if actual_col and actual_col in df.columns:
                employment_data[field] = df[actual_col]
            else:
                employment_data[field] = pd.Series([None] * len(df))
        
        # Check for valid employment records
        valid_mask = (
            employment_data['Employing Organization Name'].notna() & 
            (employment_data['Employing Organization Name'].astype(str).str.strip() != '') &
            employment_data['Employment Start Date'].notna() & 
            (employment_data['Employment Start Date'].astype(str).str.strip() != '')
        )
        
        if valid_mask.any():
            # Create employment dataframe with same index as original df
            employment_df = pd.DataFrame(employment_data, index=df.index)
            
            # Get person data with same index
            person_data = df[person_cols].copy()
            
            # Combine person and employment data
            combined = pd.concat([person_data, employment_df], axis=1)
            
            # Only keep rows with valid employment data
            valid_combined = combined[valid_mask].copy()
            
            if len(valid_combined) > 0:
                results.append(valid_combined)
    
    # Combine all results
    if results:
        final_df = pd.concat(results, ignore_index=True)
        return final_df
    else:
        # Return empty dataframe with correct columns if no valid records
        all_cols = person_cols + list(employment_groups[0].keys())
        return pd.DataFrame(columns=all_cols)
    
df = read_tbl()

df = df.pipe(normalize_employment_data)

def rename_cols(df):
    df = df.rename(columns={"Person Student ID": "uid", 
                            "Person First Name": "first_name",
                            "Person Middle Name": "middle_name",
                            "Person Last Name": "last_name",
                            "Person Suffix": "suffix",
                            "Person Gender": "sex",
                            "Person Referencing Gender": "reference_gender", 
                            "Person Status": "employment_status", 
                            "Employment Start Date": "start_date",
                            "Employment End Date": "end_date", 
                            "Employment Appointment Type": "employment_type",
                            "Employment Employment Assignment": "employment_assignment",
                            "Employment Title/Rank (Current)": "rank",
                            "Employment Status": "employment_status",
                            "Employment Change Reason": "separation_reason",
                            "Employment Change Comment": "separation_reason_desc", 
                            "Is Primary Employment": "primary_employment",
                            "Employing Organization Name": "agency_name",
                            "Employing Organization ORI or WA UBI": "agency_ori"

                            })
    
    return df.reset_index().drop(columns=["index"])


df = df.sort_values("Person Student ID")

df = df.pipe(rename_cols)

df

/var/folders/r9/3_1rmy995xs_9z4vz66rsf9r0000gn/T/ipykernel_2578/1052503451.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,163,164,165,166,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,194,195,196,197,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,245,246,247,248,249,250,251,252,253,254,255,256,257,258,260,261,

,uid,first_name,middle_name,last_name,suffix,sex,reference_gender,employment_status,start_date,end_date,employment_type,employment_assignment,rank,employment_status,separation_reason,separation_reason_desc,primary_employment,agency_name,agency_ori
0,0002-0685,Robert,J.,Douglas,NaN,Male,Male,Inactive,1/1/1901,1/1/1901,Peace Officer,NaN,Officer,Separated,Separation,NaN,True,La Conner Police Department,NaN
1,0002-0685,Robert,J.,Douglas,NaN,Male,Male,Inactive,11/15/2010,2/6/2014,Non-Certified Tribal Police Officer,NaN,Sergeant,Separated,Resignation,NaN,False,Lummi Nation Police Department,WADI05900
2,0002-1027,Elizabeth,D.,Friend,NaN,Female,Female,Active,2/5/2007,2/4/2008,Certified Peace Officer,NaN,Officer,Separated,Terminated,NaN,False,Sultan Police Department,WA0311500
3,0002-1027,Elizabeth,D.,Friend,NaN,Female,Female,Active,4/14/1997,12/11/2006,Certified Peace Officer,NaN,Officer,Separated,Separation,"NOS says ""Involuntary"". VNM",False,Bellevue Police Department,WA0170200
4,0002-1027,Elizabeth,D.,Friend,NaN,Female,Female,Active,10/20/2018,NaN,Private Security Personnel,NaN,Security Officer,Active,Hire,NaN,True,Puget Sound Executive Services,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52377,9995-0749,Edward,M.,Bush,NaN,Male,Male,Inactive,9/16/2013,NaN,Juvenile Corrections Personnel,NaN,Juvenile Corrections Officer,Active,Hire,NaN,False,Okanogan County Juvenile And Family Services,NaN
52378,9995-0749,Edward,M.,Bush,NaN,Male,Male,Inactive,8/30/2021,NaN,Juvenile Probation Personnel,NaN,Court Services Officer,Active,Hire,NaN,True,Okanogan County Juvenile And Family Services,NaN
52379,9998-0328,Richard,B.,Drake,NaN,Male,Male,Inactive,9/20/2004,8/12/2005,NaN,NaN,Unknown,Separated,Separation,NaN,True,Mason County Sheriff's Office,WA0230000
52380,9998-2062,Jason,M.,Moore,NaN,NaN,NaN,Active,11/1/1996,5/13/2006,Certified Peace Officer,NaN,Officer,Separated,Resignation,NaN,False,Cheney Police Department,WA0320100


In [28]:
# df.tail(25).to_csv("../data/output/test.csv", index=False)

In [29]:
df.columns.tolist()

['uid',
 'first_name',
 'middle_name',
 'last_name',
 'suffix',
 'sex',
 'reference_gender',
 'employment_status',
 'start_date',
 'end_date',
 'employment_type',
 'employment_assignment',
 'rank',
 'employment_status',
 'separation_reason',
 'separation_reason_desc',
 'primary_employment',
 'agency_name',
 'agency_ori']